In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR

# =============================
# Config
# =============================
DATA_PATH = "./merged_var_input.csv"
OUT_SCORES = "./lag_selection_scores.csv"
OUT_CHOICE = "./lag_selection_choice.csv"

MAX_LAGS = 10  # 필요 시 20까지도 가능 (표본/변수 수에 따라)
USE_DIFF_ONLY = True  # diff 변수만으로 VAR 할 경우 True

# =============================
# 1) Load
# =============================
df = pd.read_csv(DATA_PATH)

# Date 제거(있으면)
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.drop(columns=["Date"])

# =============================
# 2) Select columns
# =============================
if USE_DIFF_ONLY:
    cols = [c for c in df.columns if c.startswith("dlog_") or c.startswith("d_")]
else:
    # level도 포함하고 싶다면 여기서 조정
    cols = [c for c in df.columns if c not in ["Date"]]

data = df[cols].dropna()

if data.shape[0] < 50:
    raise ValueError(f"표본이 너무 적습니다. n={data.shape[0]}")

print("VAR 입력 변수:", cols)
print("표본 수:", data.shape[0])
print("변수 수:", data.shape[1])

# =============================
# 3) Fit VAR and compute IC
# =============================
model = VAR(data)

scores = []
# p=0은 보통 의미가 약하므로 1부터
for p in range(1, MAX_LAGS + 1):
    try:
        res = model.fit(p)
        scores.append({
            "lag": p,
            "aic": res.aic,
            "bic": res.bic,
            "hqic": res.hqic,
            "fpe": res.fpe
        })
    except Exception as e:
        # 특정 p에서 수치문제가 나면 기록하고 건너뜀
        scores.append({
            "lag": p,
            "aic": np.nan,
            "bic": np.nan,
            "hqic": np.nan,
            "fpe": np.nan,
        })
        print(f"[WARN] p={p} fit 실패: {e}")

scores_df = pd.DataFrame(scores)
scores_df.to_csv(OUT_SCORES, index=False)

# =============================
# 4) Choose p by each criterion
# =============================
def argmin_safe(s: pd.Series):
    s2 = s.dropna()
    return int(s2.idxmin()) if len(s2) else None

# idxmin은 row index이므로 lag 값으로 변환
best = {}
for crit in ["aic", "bic", "hqic", "fpe"]:
    idx = argmin_safe(scores_df[crit])
    best[f"p_{crit}"] = int(scores_df.loc[idx, "lag"]) if idx is not None else None

choice_df = pd.DataFrame([{
    "selected_by": "min_value",
    "p_aic": best["p_aic"],
    "p_bic": best["p_bic"],
    "p_hqic": best["p_hqic"],
    "p_fpe": best["p_fpe"],
    "max_lags": MAX_LAGS,
    "nobs_used": int(data.shape[0]),
    "n_vars": int(data.shape[1]),
}])
choice_df.to_csv(OUT_CHOICE, index=False)

print("저장 완료:", OUT_SCORES, OUT_CHOICE)
print("선택 결과:", best)

VAR 입력 변수: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']
표본 수: 1325
변수 수: 5
저장 완료: ./lag_selection_scores.csv ./lag_selection_choice.csv
선택 결과: {'p_aic': 1, 'p_bic': 1, 'p_hqic': 1, 'p_fpe': 1}
